# Unsupervised Learning: All Life Bank

## Context

AllLife Bank wants to focus on its credit card customer base in the next financial year. They have been advised by their marketing research team, that the penetration in the market can be improved. Based on this input, the Marketing team proposes to run personalized campaigns to target new customers as well as upsell to existing customers. Another insight from the market research was that the customers perceive the support services of the back poorly. Based on this, the Operations team wants to upgrade the service delivery model, to ensure that customer queries are resolved faster. The Head of Marketing and Head of Delivery both decide to reach out to the Data Science team for help.

## Objective

To identify different segments in the existing customers, based on their spending patterns as well as past interaction with the bank, using clustering algorithms, and provide recommendations to the bank on how to better market to and service these customers.


## Data Description

The data provided is of various customers of a bank and their financial attributes like credit limit, the total number of credit cards the customer has, and different channels through which customers have contacted the bank for any queries (including visiting the bank, online, and through a call center).

## Data Dictionary

* Sl_No: Primary key of the records
* Customer Key: Customer identification number
* Average Credit Limit: Average credit limit of each customer for all credit cards
* Total credit cards: Total number of credit cards possessed by the customer
* Total visits bank: Total number of visits that the customer made (yearly) personally to the bank
* Total visits online: Total number of visits or online logins made by the customer (yearly)
* Total calls made: Total number of calls made by the customer to the bank or its customer service department (yearly)


## Importing necessary libraries

In [ ]:
# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# Libraries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)

# to scale the data using z-score
from sklearn.preprocessing import StandardScaler

# to compute distances
from scipy.spatial.distance import cdist, pdist

# to perform k-means clustering and compute silhouette scores
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# to visualize the elbow curve and silhouette scores
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer

# to perform hierarchical clustering, compute cophenetic correlation, and create dendrograms
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage, cophenet

# to perform PCA
from sklearn.decomposition import PCA

# to suppress warnings
import warnings
warnings.filterwarnings("ignore")

## Loading the dataset

In [ ]:
data = pd.read_excel('Credit Card Customer Data.xlsx')

## Overview of the Dataset

In [ ]:
data.head()

In [ ]:
data.tail()

### Checking the shape of the dataset

In [ ]:
print(f"There are {data.shape[0]} rows and {data.shape[1]} columns.")

### Displaying few rows of the dataset

In [ ]:
data.sample(n=10, random_state=1)

### Checking the data types of the columns for the dataset

In [ ]:
data.info()

### Creating a copy of original data

In [ ]:
df = data.copy()

### Checking for duplicates and missing values

In [ ]:
df.duplicated().sum()

### Checking for missing values in the data

In [ ]:
df.isnull().sum()

### Statistical summary of the dataset

In [ ]:
df.drop(['Sl_No','Customer Key'],axis=1,inplace=True)

In [ ]:
df.describe(include='all').T

Customer Key:

The customers in this dataset have unique keys ranging from 11,265 to 99,843.

Avg_Credit_Limit:

The average credit limit across customers varies widely, from 3,000 to 200,000. The mean credit limit is 34,574.24, and the standard deviation is quite high at 37,625.49, indicating a large variation in credit limits. The middle 50% of customers have credit limits between 10,000 and 48,000.

Total_Credit_Cards:

Customers have between 1 to 10 credit cards, with an average of 4.71 cards. The standard deviation is 2.17, suggesting a moderate variation in the number of credit cards held. Most customers have between 3 and 6 credit cards.

Total_visits_bank:

Bank visits by customers range from 0 to 5, with an average of 2.4 visits. The standard deviation is 1.63, indicating some variation in visit frequency. The middle 50% of customers visit the bank between 1 to 4 times.

Total_visits_online:

Online visits range from 0 to 15, with a mean of 2.61 visits. The standard deviation is 2.94, suggesting a significant variation in online visit frequency. The central 50% of customers visit online between 1 to 4 times.

Total_calls_made:

Customers make between 0 to 10 calls, with an average of 3.58 calls. The standard deviation is 2.87, showing a wide variation in the number of calls made. Most customers make between 1 and 5 calls.

## Exploratory Data Analysis

### Univariate analysis

In [ ]:
# function to plot a boxplot and a histogram along the same scale.


def histogram_boxplot(df, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=df, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a star will indicate the mean value of the column
    sns.histplot(
        data=df, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=df, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        df[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        df[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        order=data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

**`Avg_Credit_Limit`**

In [ ]:
histogram_boxplot(df, 'Avg_Credit_Limit')

* The average credit limit data is highly skewed to the right, indicating most individuals have lower credit limits, with a few having significantly higher limits.
* Several outliers suggest there are individuals with exceptionally high credit limits compared to the rest.
* The mean is higher than the median, typical in a right-skewed distribution.

**`Total_Credit_Cards`**

In [ ]:
labeled_barplot(df, 'Total_Credit_Cards', perc=True)

* The majority of individuals hold 4 credit cards (22.9%).
* The second most common is 6 credit cards (17.7%).
* The least common are 8 and 9 credit cards, both at 1.7%.

**`Total_visits_bank`**

In [ ]:
labeled_barplot(df, 'Total_visits_bank', perc=True)

* The largest proportion of individuals visited the bank twice (23.9%).
* The second most common is 1 visit (17.0%).
* The least common are 0 and 3 visits, both at 15.2%.

**`Total_visits_online`**

In [ ]:
labeled_barplot(df, 'Total_visits_online', perc=True)

* The largest proportion of individuals had 2 online visits (28.6%).
* The second most common is 0 visits (21.8%).
* A significant percentage had 4 or 5 visits (10.5% and 8.2%, respectively).
* Any number of visits beyond 5 accounted for a much smaller percentage, each below 2%, with some below 1%.

**`Total_calls_made`**

In [ ]:
labeled_barplot(df, 'Total_calls_made', perc=True)

* The largest proportion of individuals made 4 calls (16.4%).
* The second most common is 2 calls (13.8%)
* A significant percentage made 0, 1, and 3 calls (14.7%, 13.6%, and 12.6%, respectively).
* Any number of calls beyond 4 accounted for a much smaller percentage, each below 6%, with some below 4%.

### Bivariate Analysis

In [ ]:
# correlation check
plt.figure(figsize=(15, 7))
sns.heatmap(
    df.corr(numeric_only = True), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral"
)
plt.show()

### Observations

**Avg_Credit_Limit:**

* Positively correlated with Total_Credit_Cards (0.61) and Total_visits_online (0.55).

* Negatively correlated with Total_visits_bank (-0.10) and Total_calls_made (-0.41).

**Total_Credit_Cards:**

* Positively correlated with Total_visits_bank (0.32) and Total_visits_online (0.17).

* Negatively correlated with Total_calls_made (-0.65).

**Total_visits_bank:**

* Negatively correlated with Total_visits_online (-0.55) and Total_calls_made (-0.51).

**Total_visits_online:**

* Positively correlated with Avg_Credit_Limit (0.55) and Total_Credit_Cards (0.17).

* Negatively correlated with Total_visits_bank (-0.55).

**Total_calls_made:**

* Negatively correlated with Avg_Credit_Limit (-0.41) and Total_Credit_Cards (-0.65).

* Negatively correlated with Total_visits_bank (-0.51).

**Inferences:**

* Individuals with a higher average credit limit tend to have more credit cards and make more online visits, but fewer bank visits and calls.

* More credit cards are associated with more bank visits but fewer calls.

* People who visit the bank more frequently tend to have fewer online visits and make fewer calls.

* A higher number of online visits is linked to a higher average credit limit and more credit cards but fewer bank visits.

* More calls are generally associated with a lower average credit limit, fewer credit cards, and fewer bank visits.


In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

In [ ]:
sns.pairplot(data=df[numeric_columns], diag_kind="kde")
plt.show()

### Observations

**Avg_Credit_Limit:**

The distribution is right-skewed, with most values concentrated below 50,000.

Positive correlation with Total_Credit_Cards, suggesting individuals with higher credit limits tend to have more credit cards.

No clear correlation with Total_visits_bank, Total_visits_online, or Total_calls_made.

**Total_Credit_Cards:**

The distribution is somewhat uniform, with a slight concentration around 2 to 4 cards.

No clear correlation with Total_visits_bank, Total_visits_online, or Total_calls_made.

**Total_visits_bank:**

Distribution peaks around 2 visits.

No clear correlation with Total_visits_online or Total_calls_made.

**Total_visits_online:**

Distribution peaks around 10 visits.

No clear correlation with Total_calls_made.

**Total_calls_made:**

Distribution peaks around 2 calls.

No clear correlation with any other variable.

## Data Preprocessing

### Outlier Check

In [ ]:
plt.figure(figsize=(15, 12))

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

for i, variable in enumerate(numeric_columns):
    plt.subplot(3, 4, i + 1)
    plt.boxplot(df[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

In [ ]:
# Scaling the data set before clustering
scaler = StandardScaler()
subset = df[numeric_columns].copy()
subset_scaled = scaler.fit_transform(subset)

In [ ]:
subset_scaled_df = pd.DataFrame(subset_scaled, columns=subset.columns)

subset_scaled_df

In [ ]:
# creating dataframe copies for k-means and hierarchical clustering
km_df = df[numeric_columns]
hc_df = df[numeric_columns]

## K-means Clustering

In [ ]:
clusters = range(1, 9)
meanDistortions = []

for k in clusters:
    model = KMeans(n_clusters=k)
    model.fit(subset_scaled_df)
    prediction = model.predict(subset_scaled_df)
    distortion = (
        sum(
            np.min(cdist(subset_scaled_df, model.cluster_centers_, "euclidean"), axis=1)
        )
        / subset_scaled_df.shape[0]
    )

    meanDistortions.append(distortion)

    print("Number of Clusters:", k, "\tAverage Distortion:", distortion)

plt.plot(clusters, meanDistortions, "bx-")
plt.xlabel("k")
plt.ylabel("Average Distortion")
plt.title("Selecting k with the Elbow Method", fontsize=20)

**The appropriate value of k from the elbow curve seems to be 3 or 4.**

In [ ]:
model = KMeans(random_state=1)
visualizer = KElbowVisualizer(model, k=(1, 9), timings=True)
visualizer.fit(subset_scaled_df)  
visualizer.show() 

### Let's check the silhouette scores

In [ ]:
sil_score = []
cluster_list = list(range(2, 10))
for n_clusters in cluster_list:
    clusterer = KMeans(n_clusters=n_clusters)
    preds = clusterer.fit_predict((subset_scaled_df))
    # centers = clusterer.cluster_centers_
    score = silhouette_score(subset_scaled_df, preds)
    sil_score.append(score)
    print("For n_clusters = {}, silhouette score is {}".format(n_clusters, score))

plt.plot(cluster_list, sil_score)

**From the silhouette scores, it seems that 3 is a good value of k.**

In [ ]:
visualizer = SilhouetteVisualizer(KMeans(3, random_state=1))
visualizer.fit(subset_scaled_df)
visualizer.show()

In [ ]:
visualizer = SilhouetteVisualizer(KMeans(4, random_state=1))
visualizer.fit(subset_scaled_df)
visualizer.show()

In [ ]:
visualizer = SilhouetteVisualizer(KMeans(5, random_state=1))
visualizer.fit(subset_scaled_df)
visualizer.show()

### Creating Final Model

In [ ]:
# final K-means model
kmeans = KMeans(n_clusters=3, random_state=1)  
kmeans.fit(subset_scaled_df)

In [ ]:


# adding kmeans cluster labels to the original and scaled dataframes
km_df["KM_segments"] = kmeans.labels_

## Cluster Profiling: K-Means Clustering

In [ ]:
km_cluster_profile = km_df.groupby("KM_segments").mean()  

In [ ]:
km_cluster_profile["count_in_each_segment"] = (
    km_df.groupby("KM_segments")["Total_Credit_Cards"].count().values  
)

In [ ]:
km_cluster_profile.style.highlight_max(color="lightgreen", axis=0)

In [ ]:
for cl in km_df["KM_segments"].unique():
    print("In cluster {}, the following groups of customers based on total credit cards owned:".format(cl))
    print(km_df[km_df["KM_segments"] == cl]["Total_Credit_Cards"].unique())
    print()

In [ ]:
km_df.groupby(["KM_segments", "Total_Credit_Cards"])['Avg_Credit_Limit'].sum()

In [ ]:
plt.figure(figsize=(20, 20))
plt.suptitle("Boxplot of numerical variables for each cluster")

# selecting numerical columns
num_col = df.select_dtypes(include=np.number).columns.tolist()

for i, variable in enumerate(num_col):
    plt.subplot(3, 4, i + 1)
    sns.boxplot(data=km_df, x="KM_segments", y=variable)

plt.tight_layout(pad=2.0)

### Insights

### Cluster 0 Behavior

**Financial Activity:**

Lowest Average Credit Limit: Individuals in Cluster 0 have the lowest average credit limit. This indicates that they might have lower financial standing or limited access to high credit limits. It could also suggest they are more conservative in their borrowing and spending habits.

Fewer Credit Cards: They have the fewest total credit cards among the clusters. This suggests a preference for maintaining a smaller number of credit card accounts, possibly due to lower credit limits or a desire to keep their financial management simpler and more manageable.

**Banking Preferences:**

Highest In-Person Bank Visits: Cluster 0 individuals have the highest number of total visits to the bank. This shows a strong preference for in-person banking activities, likely for transactions that require physical presence or for seeking assistance from bank staff. It could also indicate a lower comfort level with online banking services.

Fewest Online Visits: They have the fewest total online visits, indicating a limited adoption of digital banking services. This could be due to various factors such as lack of access to digital devices, limited internet connectivity, or simply a preference for traditional banking methods.

**Communication Channels:**

Highest Phone Interactions: Individuals in Cluster 0 make the highest number of total calls. This suggests a reliance on telephone communication for their banking needs. They might find it convenient to resolve issues, get information, or conduct transactions over the phone rather than using online services.

### Cluster 1 Behavior

**Financial Activity:**

Moderate Average Credit Limit: Individuals in Cluster 1 have a moderate average credit limit, indicating a balanced level of financial standing. They are likely to have access to reasonable credit limits without being as financially well-off as those in Cluster 2.

Moderate Number of Credit Cards: They possess a moderate number of total credit cards, suggesting a balanced approach to credit usage. They might be comfortable managing several credit accounts but without overextending themselves.

**Banking Preferences:**

Moderate In-Person Bank Visits: Cluster 1 individuals visit the bank a moderate number of times, indicating a balanced approach to in-person banking. They utilize physical banking services as needed but are not overly reliant on them.

Moderate Online Visits: They also have a moderate number of online visits, reflecting a reasonable adoption of digital banking services. This suggests that they are comfortable with both traditional and digital methods of managing their finances.

**Communication Channels:**

Moderate Phone Interactions: Individuals in Cluster 1 make a moderate number of phone calls. This balanced approach indicates that they use phone communication when necessary but also rely on other methods such as in-person and online interactions.


### Cluster 2 Behavior

**Financial Activity:**

Highest Average Credit Limit: Individuals in Cluster 2 have the highest average credit limit, indicating they have a strong financial standing and access to higher credit limits. This suggests a high level of creditworthiness and financial responsibility.

Most Credit Cards: They hold the most credit cards compared to the other clusters. This indicates that they are comfortable managing multiple credit accounts, and likely have a diverse portfolio of credit options.

**Banking Preferences:**

Fewest In-Person Bank Visits: Cluster 2 individuals have the fewest total visits to the bank. This suggests a strong preference for avoiding in-person banking, potentially due to their comfort and familiarity with digital banking services.

Highest Online Visits: They have the highest number of online visits, indicating a strong preference for digital banking. This suggests that they are tech-savvy, comfortable with online transactions, and prefer the convenience of managing their finances online.

**Communication Channels:**

Fewest Phone Calls: Individuals in Cluster 2 make the fewest phone calls. This indicates a preference for handling their banking activities through online channels rather than over the phone. They might find online platforms more efficient and convenient for their needs.




## Hierarchical Clustering

### Computing Cophenetic Correlation

In [ ]:
subset_scaled_df

In [ ]:
# list of distance metrics
distance_metrics = ["euclidean", "chebyshev", "mahalanobis", "cityblock"]

# list of linkage methods
linkage_methods = ["single", "complete", "average", "weighted"]

high_cophenet_corr = 0
high_dm_lm = [0, 0]

for dm in distance_metrics:
    for lm in linkage_methods:
        Z = linkage(subset_scaled_df, metric=dm, method=lm)
        c, coph_dists = cophenet(Z, pdist(subset_scaled_df))
        print(
            "Cophenetic correlation for {} distance and {} linkage is {}.".format(
                dm.capitalize(), lm, c
            )
        )
        if high_cophenet_corr < c:
            high_cophenet_corr = c
            high_dm_lm[0] = dm
            high_dm_lm[1] = lm

In [ ]:
# printing the combination of distance metric and linkage method with the highest cophenetic correlation
print(
    "Highest cophenetic correlation is {}, which is obtained with {} distance and {} linkage.".format(
        high_cophenet_corr, high_dm_lm[0].capitalize(), high_dm_lm[1]
    )
)

### Checking Dendrograms

**We see that the cophenetic correlation is maximum with Euclidean distance and average linkage.**


**Let's view the dendrograms for the different linkage methods.**

In [ ]:
# list of linkage methods
linkage_methods = ["single", "complete", "average", "centroid", "ward", "weighted"]

# lists to save results of cophenetic correlation calculation
compare_cols = ["Linkage", "Cophenetic Coefficient"]
compare = []

# to create a subplot image
fig, axs = plt.subplots(len(linkage_methods), 1, figsize=(15, 30))

# We will enumerate through the list of linkage methods above
# For each linkage method, we will plot the dendrogram and calculate the cophenetic correlation
for i, method in enumerate(linkage_methods):
    Z = linkage(subset_scaled_df, metric="euclidean", method=method)

    dendrogram(Z, ax=axs[i])
    axs[i].set_title(f"Dendrogram ({method.capitalize()} Linkage)")

    coph_corr, coph_dist = cophenet(Z, pdist(subset_scaled_df))
    axs[i].annotate(
        f"Cophenetic\nCorrelation\n{coph_corr:0.2f}",
        (0.80, 0.80),
        xycoords="axes fraction",
    )

    compare.append([method, coph_corr])

### **Observations**

- Looking the the above dendrograms, the Ward linkage seems to result in the best separation between clusters, even though its cophenetic correlation is lower than the other linkages.
- 3 looks to be a good choice for no. of clusters.

### Creating Final Model

In [ ]:
hc = AgglomerativeClustering(n_clusters=3, metric="euclidean", linkage="ward")
hc_labels = hc.fit_predict(hc_df)

In [ ]:
hc_df["HC_segments"] = hc_labels

In [ ]:
# adding hc cluster labels to the original and scaled dataframes
hc_df["HC_segments"] = hc.labels_


In [ ]:
hc_df

## Cluster Profiling: Hierarchical Clustering

In [ ]:
pd.crosstab(hc_df.HC_segments, hc_df.Total_Credit_Cards)

In [ ]:
clusters = hc_df.HC_segments.unique().tolist()
for cl in clusters:
    print(
        "The",
        hc_df[hc_df["HC_segments"] == cl]["Total_Credit_Cards"].nunique(),
        "group of customers in cluster",
        cl,
        "are:",
    )
    print(hc_df[hc_df["HC_segments"] == cl]["Total_Credit_Cards"].unique())
    print("-" * 100, "\n")

In [ ]:
for cl in hc_df["HC_segments"].unique():
    print("In cluster {}, the following groups of customers based on total credit cards owned:".format(cl))
    print(hc_df[hc_df["HC_segments"] == cl]["Total_Credit_Cards"].unique())
    print()


In [ ]:
hc_df.groupby(["HC_segments", "Total_Credit_Cards"])['Avg_Credit_Limit'].sum()

In [ ]:
hc_cluster_profile = hc_df.groupby("HC_segments").mean()  

In [ ]:
hc_cluster_profile["count_in_each_segment"] = (
    hc_df.groupby("HC_segments")["Total_Credit_Cards"].count().values  
)

In [ ]:
# displaying the group-wise means of variables
hc_cluster_profile.style.highlight_max(color="lightgreen", axis=0)

In [ ]:
plt.figure(figsize=(20, 35))
plt.suptitle("Boxplot of scaled numerical variables for each cluster", fontsize=20)

# selecting numerical columns
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

for i, variable in enumerate(numeric_columns):
    plt.subplot(5, 3, i + 1)
    sns.boxplot(data=hc_df, x="HC_segments", y=variable)

plt.tight_layout(pad=2.0)

### Insights

#### Cluster 0 Behavior

**Financial Activity:**

Lower Average Credit Limit: Individuals in Cluster 0 have the lowest average credit limit, indicating that they might have lower financial standing or limited access to high credit limits.

Fewer Credit Cards: They also have fewer credit cards compared to individuals in other clusters. This suggests that they might prefer to keep their credit card usage to a minimum, possibly to avoid debt or due to limited credit availability.

**Banking Preferences:**

Moderate In-Person Banking: Cluster 0 individuals have a moderate number of total visits to the bank. This shows that they engage in in-person banking activities fairly regularly, possibly for transactions that require physical presence or for consulting bank staff.

Low Online Visits: They have the least online visits among the clusters. This indicates a preference for traditional banking methods over digital interactions, possibly due to limited access to online banking facilities, a lack of trust in online systems, or simply a preference for face-to-face interactions.

**Communication Channels:**

High Phone Interactions: Individuals in Cluster 0 make the highest number of phone calls. This reliance on telephone communication suggests that they prefer handling their banking activities through phone calls rather than online or in-person visits. They might find it convenient to resolve issues, get information, or conduct transactions over the phone.


#### Cluster 1 Behavior

**Financial Activity:**

Moderate Average Credit Limit: Individuals in Cluster 1 have a moderate average credit limit, which suggests they have a reasonable level of financial standing, falling between the higher limits of Cluster 2 and the lower limits of Cluster 0.

Moderate Number of Credit Cards: They also hold a moderate number of total credit cards, which indicates a balanced approach to credit usage, not as conservative as Cluster 0 but not as extensive as Cluster 2.

**Banking Preferences:**

Moderate In-Person Banking: Cluster 1 individuals have a moderate number of total visits to the bank, similar to Cluster 0. This suggests a balanced approach to in-person banking, engaging in it as needed but not excessively.

Moderate Online Visits: They also have a moderate number of online visits, indicating a reasonable adoption of digital banking services. They are likely comfortable with both traditional and digital methods of banking.

**Communication Channels:**

Moderate Phone Interactions: Cluster 1 individuals make a moderate number of phone calls. This balanced approach to phone interactions suggests that they use this communication channel when necessary but also rely on other methods such as in-person and online interactions.

#### Cluster 2 Behavior

**Financial Activity:**

Highest Average Credit Limit: Individuals in Cluster 2 have the highest average credit limit, indicating they have the strongest financial standing and access to higher credit limits.

Most Credit Cards: They hold the most credit cards compared to the other clusters, suggesting they are comfortable managing multiple credit accounts and likely have a high level of financial responsibility.

**Banking Preferences:**

Fewest In-Person Bank Visits: Cluster 2 individuals have the lowest number of total visits to the bank. This indicates a preference for avoiding in-person banking and possibly relying more on other banking methods.

Highest Online Visits: They have the highest number of online visits, showing a strong preference for digital banking services. This suggests that they are tech-savvy and comfortable managing their finances online.

**Communication Channels:**

Fewest Phone Calls: Individuals in Cluster 2 make the fewest phone calls. This indicates that they prefer to handle their banking activities through online channels rather than over the phone.



## K-means vs Hierarchical Clustering

**Observations on Cluster 0**

* Both methods indicate Cluster 0 individuals have lower financial standing and fewer credit cards.

* Hierarchical Clustering shows moderate in-person banking, while K-means indicates the highest bank visits.

* Both methods agree on low online activity and high phone interactions.


**Observations on Cluster 1**

* Both methods consistently characterize Cluster 1 with moderate financial standing, balanced credit card usage, and moderate banking preferences.

* Both methods show moderate in-person and online activity and balanced phone interactions.

**Observations on Cluster 2**

* Both methods agree that Cluster 2 individuals have the highest financial standing and most credit cards.

* Both indicate a preference for online banking and the fewest in-person visits and phone calls.

**Overall Comparison:**

**Summary of Comparisons:**
**Cluster 0:**

* K-means: Lower financial standing, highest bank visits, lowest online visits, highest phone calls.

* Hierarchical: Lower financial standing, moderate bank visits, low online visits, high phone calls.

**Cluster 1:**

* K-means: Moderate financial standing, balanced banking behavior.

* Hierarchical: Moderate financial standing, balanced banking behavior.

**Cluster 2:**

* K-means: Highest financial standing, preference for online banking, few in-person visits and phone calls.

* Hierarchical: Highest financial standing, preference for online banking, few in-person visits and phone calls.

**Insights:**
* Consistency: Both clustering methods provide consistent insights for Clusters 1 and 2, indicating agreement on moderate and high financial standing, respectively.

* Variations: There are slight variations for Cluster 0, particularly in the interpretation of in-person banking activity.



# Actionable Insights and Recommendations


**Cluster 0 (Lower Financial Standing):**
**Insights:**

* Lower average credit limit and fewer credit cards indicate limited financial capacity.

* Preference for in-person banking and high phone interactions suggests reliance on traditional banking methods.

* Low online visits indicate limited adoption of digital banking services.

**Recommendations:**

**Enhance In-Person Services:**

* Offer personalized in-person services such as appointment-based consultations.

* Provide dedicated customer service representatives to assist with their banking needs.

**Improve Phone Support:**

* Ensure responsive and knowledgeable phone support with minimal wait times.

* Provide clear and concise information through phone interactions to address their queries and issues effectively.

**Digital Adoption Initiatives:**

* Conduct workshops or educational programs to encourage digital banking adoption.

* Offer incentives such as lower fees or cashback rewards for using online banking services.

* Provide easy-to-use mobile banking apps and online platforms to build trust and familiarity.

**Cluster 1 (Moderate Financial Standing):**
**Insights:**

* Moderate average credit limit and balanced number of credit cards indicate a stable financial position.

* Balanced approach to in-person banking, online visits, and phone interactions suggests comfort with various banking methods.

**Recommendations:**

**Provide Flexible Banking Options:**

* Offer a mix of traditional and digital banking services to cater to their diverse preferences.

* Ensure seamless integration between in-person, online, and phone banking channels for a consistent customer experience.

**Targeted Marketing:**

* Promote hybrid banking accounts that combine in-person benefits with online convenience.

* Offer tailored financial products and services that meet their balanced banking behavior.

**Loyalty Programs:**

* Implement loyalty programs to reward their consistent engagement across different banking channels.

* Provide personalized offers and discounts based on their banking activities and preferences.

**Cluster 2 (Strong Financial Standing):**
**Insights:**

* Highest average credit limit and most credit cards indicate strong financial capacity and creditworthiness.

* Preference for online banking and low in-person visits suggest high digital adoption and tech-savviness.

* Few phone calls indicate a preference for self-service and online interactions.

**Recommendations:**

**Enhance Digital Services:**

* Invest in advanced digital banking platforms with a wide range of features and functionalities.

* Provide seamless and secure online banking experiences through mobile apps and web platforms.

**Virtual Customer Support:**

* Offer virtual consultations and online customer service to cater to their preference for digital interactions.

* Implement chatbots and AI-driven support systems to provide instant assistance and resolution.

**Exclusive Digital Products:**

* Launch online-only accounts and digital investment platforms tailored to their financial needs.

* Offer virtual financial planning services and personalized financial advice through digital channels.

**Promote Self-Service Options:**

* Provide comprehensive self-service options for transactions, account management, and financial planning.

* Ensure that all digital services are user-friendly and easily accessible.

**Summary:**
* Cluster 0: Focus on enhancing in-person and phone support, and encourage digital adoption through educational programs and incentives.

* Cluster 1: Provide flexible banking options, targeted marketing, and loyalty programs to reward balanced banking behavior.

* Cluster 2: Invest in advanced digital services, offer virtual customer support, promote self-service options, and launch exclusive digital products.